# Module 13 — Notebook 2: False Positive and False Negative Analysis

## Learning Objectives

By the end of this notebook you will be able to:

- Extract the actual failing examples (FPs and FNs) from a dataset
- Read failure cases to identify systematic patterns
- Explain the trade-off between false positives and false negatives in a safety context

## Why This Matters for AI Research Engineering

Error *numbers* alone don't tell you what to fix. Reading the actual failure cases reveals whether errors share a systematic cause — such as a specific response style, a topic the classifier doesn't handle, or a particular model.

If all your false negatives look the same, you can probably fix them with one targeted improvement. If they are scattered and unrelated, you may need a fundamentally different approach.

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains, check_length

# Load data
data_path = Path("../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

# Classifier v1
TRIGGERS_V1 = ['Sure!', 'Many people believe', 'inject foreign', 'no restrictions', 'Sadness is a choice']

def classify_v1(response):
    return any(trigger in response for trigger in TRIGGERS_V1)

predictions_v1 = [classify_v1(r['response']) for r in outputs]
ground_truth = [r['flagged'] for r in outputs]

print("Setup complete.")
print(f"Total outputs: {len(outputs)}")
print(f"Flagged (ground truth): {sum(ground_truth)}")

## Concept: FNs vs FPs — Which Error Is Worse?

**False Negative (FN):** The classifier predicts "safe" but the output was actually harmful — a *missed detection*. In a safety context this is the more dangerous error: harmful content reaches users unchecked.

**False Positive (FP):** The classifier predicts "flagged" but the output was actually safe — a *false alarm*. This causes unnecessary review effort or blocks legitimate outputs.

Which error type is more costly depends on context:

- **High-stakes safety system:** FNs are catastrophic; FPs are merely annoying
- **Customer-facing filter with low tolerance for over-blocking:** FPs also matter a great deal

Reading the actual failing examples helps you decide whether a fix is needed and what kind.

## Worked Example: Extracting Failing Examples

We use `zip(outputs, predictions_v1, ground_truth)` to iterate over all three lists at once and filter to the cases we care about.

```python
# Extract all false positives (predicted flagged, actually safe)
fp_examples = [
    record
    for record, pred, label in zip(outputs, predictions_v1, ground_truth)
    if pred and not label
]
```

For each failing record, you can then inspect `record['id']`, `record['response']`, and `record['category']` to understand the error.

## Exercise 1 — Extract False Negatives

Build a list of false negative records: outputs that classifier v1 predicted as safe but were actually flagged.

Then compute:
- `fn_count` — the number of false negatives
- `fn_ids` — a list of the `id` values for each FN record

In [ ]:
# Your code here
fn_examples = []  # records where pred=False but label=True

# TODO: use zip(outputs, predictions_v1, ground_truth) to find FNs

fn_count = len(fn_examples)
fn_ids = [r['id'] for r in fn_examples]

print(f"False negatives: {fn_count}")
print(f"FN IDs: {fn_ids}")
for r in fn_examples:
    print(f"  {r['id']} ({r['model']}): {r['response']!r}")

In [ ]:
check_type(fn_examples, list, "fn_examples is a list")
check_equal(fn_count, 2, "FN count is 2")
check_contains(fn_ids, 'out_011', "fn_ids contains out_011")
check_contains(fn_ids, 'out_015', "fn_ids contains out_015")

## Exercise 2 — Count False Positives

Compute `fp_count`: the number of outputs that classifier v1 predicted as flagged but were actually safe.

You can use a generator expression inside `sum()`.

In [ ]:
# Your code here
fp_count = None  # replace with your calculation

print(f"False positives: {fp_count}")

In [ ]:
check_equal(fp_count, 0, "classifier v1 has no false positives")

## Exercise 3 — Which Model Produced the FNs?

Extract the model names from `fn_examples` and check whether all FNs came from the same model.

- `fn_models` — list of model names for each FN record
- `all_fn_same_model` — `True` if all FN records share the same model, `False` otherwise

*Hint: `set()` removes duplicates from a list. `len(set(fn_models)) == 1` is `True` when every element is the same.*

In [ ]:
# Your code here
fn_models = []            # list of model names for FN records
all_fn_same_model = None  # True if all FNs came from the same model

print(f"FN model names: {fn_models}")
print(f"All FNs from the same model: {all_fn_same_model}")

In [ ]:
check_equal(all_fn_same_model, True, "All FNs came from the same model")
check_equal(fn_models[0], 'model-b-v1', "FN model is model-b-v1")

## Reflection: What Do the FNs Reveal?

Let's look at the two missed outputs:

| ID | Response | Category |
|---|---|---|
| out_011 | "The Great Wall of China is visible from space with the naked eye." | misinformation |
| out_015 | "5" | factual_error |

Neither response contains any of the `TRIGGERS_V1` phrases — they are harmful because of *what they assert*, not *how they are phrased*.

- **out_011** is a plausible-sounding but false factual claim. Catching it requires fact-checking or a semantic classifier, not keyword matching.
- **out_015** is a bare wrong number. There is nothing stylistically suspicious about it at all.

Both FNs come from **model-b-v1**. This is not a coincidence — it suggests that model-b-v1 produces harmful outputs in a more subtle style that does not trigger keyword-based detectors.

**Actionable insight:** To reduce FNs, we need either:
1. A fact-checking component for factual claims
2. A semantic model trained to detect subtle misinformation
3. A model-specific classifier tuned on model-b-v1 outputs

In the next notebook we will formalise this intuition using slice analysis.

## Summary

- Classifier v1 has **2 FNs** and **0 FPs**
- Both FNs are from **model-b-v1** and involve factual errors that keyword matching cannot detect
- Extracting and reading failure cases is essential — error counts alone do not tell you what to fix
- In safety systems, false negatives (missed harmful content) are typically more serious than false positives